# Nonlinear Candidate Constraints

既知の入力制約 `g(X) >= 0` を、surrogate model → acquisition → candidate optimization の一連の流れで確認します。

Outcome constraint `c(Y) <= 0` とは別レイヤーです。詳細は `docs/optimization/nonlinear-candidate-constraints.md` を参照してください。

## 1. Imports and reproducibility

In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import SingleTaskGP
from robotorchan.optim import CandidateConstraints, optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(21)

## 2. Fit the surrogate

2次元の既知関数を観測したものとして `SingleTaskGP` を学習します。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    x0 = X[..., :1]
    x1 = X[..., 1:2]
    return 1.0 - 4.0 * (x0 - 0.72).square() - 3.0 * (x1 - 0.58).square()


train_X = torch.rand(16, 2)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

acquisition = qLogExpectedImprovement(model=model, best_f=train_Y.max())
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

## 3. Intra-point nonlinear constraint

中心 `(0.5, 0.5)`、半径 `0.35` の円内だけをcandidateとして許可します。BoTorch/robotorchanのcandidate constraint conventionは `g(X) >= 0` です。

In [ ]:
center = torch.tensor([0.5, 0.5])
radius = 0.35


def inside_circle(x: torch.Tensor) -> torch.Tensor:
    return x.new_tensor(radius**2) - (x - center.to(x)).square().sum()


constraints = CandidateConstraints(
    nonlinear_inequality_constraints=((inside_circle, True),),
)

initial_conditions = torch.tensor(
    [
        [[0.50, 0.50]],
        [[0.60, 0.50]],
        [[0.45, 0.60]],
        [[0.55, 0.40]],
    ]
)

candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=4,
    raw_samples=None,
    batch_initial_conditions=initial_conditions,
    constraints=constraints,
)

feasibility = inside_circle(candidate[0])
assert feasibility >= -1e-6
print("candidate:", candidate)
print("acquisition value:", value)
print("g(candidate):", feasibility)

## 4. Visualize the feasible region and candidate

In [ ]:
theta = torch.linspace(0.0, 2.0 * torch.pi, 200)
circle_x = center[0] + radius * torch.cos(theta)
circle_y = center[1] + radius * torch.sin(theta)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], label="observed")
ax.plot(circle_x, circle_y, label="g(X) = 0")
ax.scatter(candidate[:, 0], candidate[:, 1], marker="*", s=140, label="candidate")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1")
ax.legend()
plt.show()

## 5. Joint q-batch with an inter-point constraint

次は `q=2` を同時最適化し、2候補間の二乗距離を `0.12**2` 以上にします。`is_intrapoint=False` なのでcallableはjoint `[q, d]` を受け取ります。

In [ ]:
min_distance = 0.12


def separated(X: torch.Tensor) -> torch.Tensor:
    squared_distance = (X[0] - X[1]).square().sum()
    return squared_distance - X.new_tensor(min_distance**2)


batch_constraints = CandidateConstraints(
    nonlinear_inequality_constraints=(
        (inside_circle, True),
        (separated, False),
    ),
)

batch_initial_conditions = torch.tensor(
    [
        [[0.40, 0.50], [0.60, 0.50]],
        [[0.50, 0.38], [0.50, 0.62]],
        [[0.38, 0.42], [0.62, 0.58]],
        [[0.42, 0.62], [0.58, 0.38]],
    ]
)

batch_candidate, batch_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=None,
    batch_initial_conditions=batch_initial_conditions,
    constraints=batch_constraints,
)

assert all(inside_circle(x) >= -1e-6 for x in batch_candidate)
assert separated(batch_candidate) >= -1e-6
print("q=2 candidates:\n", batch_candidate)
print("joint acquisition value:", batch_value)
print("inter-point g(X):", separated(batch_candidate))

## 6. Visualize the constrained q-batch

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], label="observed")
ax.plot(circle_x, circle_y, label="intra-point boundary")
ax.scatter(
    batch_candidate[:, 0],
    batch_candidate[:, 1],
    marker="*",
    s=140,
    label="q=2 candidates",
)
ax.plot(batch_candidate[:, 0], batch_candidate[:, 1], linestyle="--")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1")
ax.legend()
plt.show()

## 7. Update the data

candidate constraintは候補生成時の既知feasibilityです。実験・シミュレーション後は通常どおり新しい `(X, Y)` をtraining dataへ追加します。

In [ ]:
new_Y = response(batch_candidate)
updated_X = torch.cat([train_X, batch_candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, new_Y.detach()], dim=0)

assert updated_X.shape == torch.Size([18, 2])
assert updated_Y.shape == torch.Size([18, 1])
print("updated shapes:", updated_X.shape, updated_Y.shape)

## 8. Contract summary

- candidate nonlinear feasibilityは `g(X) >= 0`。
- intra-point callableは `[d]`、inter-point callableはjoint `[q, d]` を受け取ります。
- nonlinear optimizationにはfeasible initial conditionsまたは適切な `ic_generator` が必要です。
- inter-point nonlinear constraintはjoint q semanticsを必要とし、greedy sequentialへ分解できません。
- candidate constraintはOutcome constraint `c(Y) <= 0` と独立して併用できます。
- robust input perturbationでは通常nominal Xを制約し、全perturbation scenarioのfeasibilityを保証するものではありません。